In [51]:
import numpy as np
from astropy.table import Table, vstack

In [83]:
tbl = Table.read('../data/B2_mssa_prep_table.fits')
tbl.sort(["timestep", "jphi_cen", "tphi_cen"])

In [90]:
tbl

timestep,jphi_cen,tphi_cen,m0_amp,m1_amp,m2_amp,pitch_ang_m1,pitch_ang_m2,phase_ang_m1,phase_ang_m2,pitch_phase_flag_m1,pitch_phase_flag_m2,time_since_int_m1,time_since_int_m2
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
250.0,1000.0,0.19634954084936207,170216.9339784259,885.519163446094,1435.043639030457,-0.09981540872292492,-0.11367870706069115,1.4242070911826232,3.2483291195331994,0.0,0.0,0.3618739634330326,0.31742785668444506
250.0,1000.0,0.5890486225480862,170806.60214148727,715.4636555405455,858.5891285586695,-0.06078363812369746,-0.44257854190813156,0.40421091125196895,0.847226990261289,0.0,0.0,0.6369430063886421,0.08179078006921717
250.0,1000.0,0.9817477042468103,171037.27607065375,813.9864298372028,900.1689040025753,-0.07543120015344605,-1.1181988466114643,0.5495620501237255,3.817583131715029,1.0,1.0,-1.0,-1.0
250.0,1000.0,1.3744467859455345,170618.4685695552,902.4044514712627,994.1099832614356,-0.06107338407894681,-1.4456991171565992,4.871166160795507,3.680581745777786,1.0,1.0,-1.0,-1.0
250.0,1000.0,1.7671458676442586,171077.70010258933,993.8493386963814,1100.1365494567603,-0.0953710628045194,-0.825563240013646,2.404486825842074,5.000265181918942,0.0,0.0,0.4159435897310065,0.0367152140872412
250.0,1000.0,2.1598449493429825,170013.12825565157,816.8904040023606,956.7464122348603,-0.12261038526300888,-0.22338545829525727,4.225474953981379,1.1871584347884774,0.0,0.0,0.30654796826926645,0.1662821117858748
250.0,1000.0,2.552544031041707,169745.2075869047,725.3724885577429,615.9877370993034,-1.5707963267948963,-0.11412690349447809,1.6197907988207563,4.54435362083596,1.0,1.0,-1.0,-1.0
250.0,1000.0,2.945243112740431,170335.46306213253,840.9185554045278,748.9830927872621,-0.08770544466598787,-1.1374707493226572,3.849039251572872,4.140370968648403,0.0,0.0,0.4262518375155744,0.017340557707017177
250.0,1000.0,3.3379421944391554,171588.68606452833,1042.8074712698888,617.7369424352264,-0.08840454243904919,-1.570796326794885,5.0142383141706315,0.8852959515232603,0.0,0.0,0.421767821365139,4.3393243230514656e-16


In [91]:
tbl['time_since_int_m1'][(tbl['pitch_phase_flag_m1'] > 0) | ~(tbl['time_since_int_m1'] > -np.inf) | ~(tbl['time_since_int_m1'] < np.inf)] = -1
tbl['time_since_int_m2'][(tbl['pitch_phase_flag_m2'] > 0) | ~(tbl['time_since_int_m2'] > -np.inf) | ~(tbl['time_since_int_m2'] < np.inf)] = -1

In [92]:
time_ratio = 0.009778

In [93]:
jphi_c = np.arange(1000, 4000+1, 100)
tphi_c_ = np.linspace(0, 2*np.pi, 16+1)
rad = [0.5*(jphi_c[1] - jphi_c[0]), 0.5*(tphi_c_[1] - tphi_c_[0])]
tphi_c = tphi_c_[:-1] + rad[1]
centers = np.array(np.meshgrid(jphi_c, tphi_c)).T.reshape(-1,2)

cfs_m0_amp    = ['m0_amp_{}_{}pi16'.format(int(cf[0]), int(16*cf[1]/np.pi)) for cf in centers]
cfs_m1_amp    = ['m1_amp_{}_{}pi16'.format(int(cf[0]), int(16*cf[1]/np.pi)) for cf in centers]
cfs_m2_amp    = ['m2_amp_{}_{}pi16'.format(int(cf[0]), int(16*cf[1]/np.pi)) for cf in centers]
cfs_m1_pitch  = ['m1_pitch_{}_{}pi16'.format(int(cf[0]), int(16*cf[1]/np.pi)) for cf in centers]
cfs_m2_pitch  = ['m2_pitch_{}_{}pi16'.format(int(cf[0]), int(16*cf[1]/np.pi)) for cf in centers]
cfs_m1_phase  = ['m1_phase_{}_{}pi16'.format(int(cf[0]), int(16*cf[1]/np.pi)) for cf in centers]
cfs_m2_phase  = ['m2_phase_{}_{}pi16'.format(int(cf[0]), int(16*cf[1]/np.pi)) for cf in centers]
cfs_m1_int_time = ['m1_int_time_{}_{}pi16'.format(int(cf[0]), int(16*cf[1]/np.pi)) for cf in centers]
cfs_m2_int_time = ['m2_int_time_{}_{}pi16'.format(int(cf[0]), int(16*cf[1]/np.pi)) for cf in centers]

colnames = ['timestep']+cfs_m0_amp+cfs_m1_amp+cfs_m2_amp+cfs_m1_pitch+cfs_m2_pitch+cfs_m1_phase+cfs_m2_phase+cfs_m1_int_time+cfs_m2_int_time

t = Table(names=colnames)

for time in np.unique(tbl['timestep']):
    subset = tbl[tbl['timestep'] == time]
    channels = np.concatenate([np.array(subset['m0_amp']), np.array(subset['m1_amp']), np.array(subset['m2_amp']),
                               np.array(subset['pitch_ang_m1']), np.array(subset['pitch_ang_m2']), 
                               np.array(subset['phase_ang_m1']), np.array(subset['phase_ang_m2']),
                               np.array(time*time_ratio - subset['time_since_int_m1']),
                               np.array(time*time_ratio - subset['time_since_int_m2'])])
    t.add_row(np.append([time], channels))

np.savetxt('../data/B2_for_mssa.dat', t)

In [98]:
def save_coefs(directory, channel_file, n_channels, j_bins=30, start_timestep=0, end_timestep=None):
    file = np.loadtxt(directory+channel_file)
    n_times = file.shape[0]
    if end_timestep==None:
        times = np.reshape(file[start_timestep:,0], (n_times-start_timestep,1))
    else:
        times = np.reshape(file[start_timestep:end_timestep,0], (end_timestep-start_timestep,1))
    
    m0_amp_coefs = file[start_timestep:end_timestep,1:n_channels+1]# - 16*5]
    m1_amp_coefs = file[start_timestep:end_timestep,n_channels+1:2*n_channels+1]# - 16*5]
    m2_amp_coefs = file[start_timestep:end_timestep,2*n_channels+1:3*n_channels+1]# - 16*5]
    m1_pitch_coefs = file[start_timestep:end_timestep,3*n_channels+1:4*n_channels+1]# - 16*5]
    m2_pitch_coefs = file[start_timestep:end_timestep,4*n_channels+1:5*n_channels+1]# - 16*5]
    m1_phase_coefs = file[start_timestep:end_timestep,5*n_channels+1:6*n_channels+1]# - 16*5]
    m2_phase_coefs = file[start_timestep:end_timestep,6*n_channels+1:7*n_channels+1]# - 16*5]
    m1_int_time_coefs = file[start_timestep:end_timestep,7*n_channels+1:8*n_channels+1]# - 16*5]
    m2_int_time_coefs = file[start_timestep:end_timestep,8*n_channels+1:9*n_channels+1]# - 16*5]
    
    m1_amp_rel_coefs = m1_amp_coefs/m0_amp_coefs
    m2_amp_rel_coefs = m2_amp_coefs/m0_amp_coefs
    
    m0_amp = np.concatenate([times, m0_amp_coefs], axis=1)
    m1_amp = np.concatenate([times, m1_amp_coefs], axis=1)
    m2_amp = np.concatenate([times, m2_amp_coefs], axis=1)
    m1_pitch = np.concatenate([times, m1_pitch_coefs], axis=1)
    m2_pitch = np.concatenate([times, m2_pitch_coefs], axis=1)
    m1_phase = np.concatenate([times, m1_phase_coefs], axis=1)
    m2_phase = np.concatenate([times, m2_phase_coefs], axis=1)
    m1_int_time = np.concatenate([times, m1_int_time_coefs], axis=1)
    m2_int_time = np.concatenate([times, m2_int_time_coefs], axis=1)
    
    m1_amp_rel = np.concatenate([times, m1_amp_rel_coefs], axis=1)
    m2_amp_rel = np.concatenate([times, m2_amp_rel_coefs], axis=1)
    
    fname_prefix = directory+'mssa_channels_B2/'
    np.savetxt(fname_prefix + 'm0_amp_bins_j{}_t16.dat'.format(j_bins), m0_amp)
    np.savetxt(fname_prefix + 'm1_amp_bins_j{}_t16.dat'.format(j_bins), m1_amp)
    np.savetxt(fname_prefix + 'm2_amp_bins_j{}_t16.dat'.format(j_bins), m2_amp)
    np.savetxt(fname_prefix + 'm1_pitch_bins_j{}_t16.dat'.format(j_bins), m1_pitch)
    np.savetxt(fname_prefix + 'm2_pitch_bins_j{}_t16.dat'.format(j_bins), m2_pitch)
    np.savetxt(fname_prefix + 'm1_amp_rel_bins_j{}_t16.dat'.format(j_bins), m1_amp_rel)
    np.savetxt(fname_prefix + 'm2_amp_rel_bins_j{}_t16.dat'.format(j_bins), m2_amp_rel)
    np.savetxt(fname_prefix + 'm1_phase_bins_j{}_t16.dat'.format(j_bins), m1_phase)
    np.savetxt(fname_prefix + 'm2_phase_bins_j{}_t16.dat'.format(j_bins), m2_phase)
    np.savetxt(fname_prefix + 'm1_int_time_bins_j{}_t16.dat'.format(j_bins), m1_int_time)
    np.savetxt(fname_prefix + 'm2_int_time_bins_j{}_t16.dat'.format(j_bins), m2_int_time)

def save_coefs_first_passage(directory, channel_file, n_channels, j_bins=30, start_timestep=0, end_timestep=210):
    file = np.loadtxt(directory+channel_file)
    n_times = file.shape[0]
    if end_timestep==None:
        times = np.reshape(file[start_timestep:,0], (n_times-start_timestep,1))
    else:
        times = np.reshape(file[start_timestep:end_timestep,0], (end_timestep-start_timestep,1))
    
    m0_amp_coefs = file[start_timestep:end_timestep,1:n_channels+1]# - 16*5]
    m1_amp_coefs = file[start_timestep:end_timestep,n_channels+1:2*n_channels+1]# - 16*5]
    m2_amp_coefs = file[start_timestep:end_timestep,2*n_channels+1:3*n_channels+1]# - 16*5]
    m1_pitch_coefs = file[start_timestep:end_timestep,3*n_channels+1:4*n_channels+1]# - 16*5]
    m2_pitch_coefs = file[start_timestep:end_timestep,4*n_channels+1:5*n_channels+1]# - 16*5]
    m1_phase_coefs = file[start_timestep:end_timestep,5*n_channels+1:6*n_channels+1]# - 16*5]
    m2_phase_coefs = file[start_timestep:end_timestep,6*n_channels+1:7*n_channels+1]# - 16*5]
    m1_int_time_coefs = file[start_timestep:end_timestep,7*n_channels+1:8*n_channels+1]# - 16*5]
    m2_int_time_coefs = file[start_timestep:end_timestep,8*n_channels+1:9*n_channels+1]# - 16*5]
    
    m1_amp_rel_coefs = m1_amp_coefs/m0_amp_coefs
    m2_amp_rel_coefs = m2_amp_coefs/m0_amp_coefs
    
    m0_amp = np.concatenate([times, m0_amp_coefs], axis=1)
    m1_amp = np.concatenate([times, m1_amp_coefs], axis=1)
    m2_amp = np.concatenate([times, m2_amp_coefs], axis=1)
    m1_pitch = np.concatenate([times, m1_pitch_coefs], axis=1)
    m2_pitch = np.concatenate([times, m2_pitch_coefs], axis=1)
    m1_phase = np.concatenate([times, m1_phase_coefs], axis=1)
    m2_phase = np.concatenate([times, m2_phase_coefs], axis=1)
    m1_int_time = np.concatenate([times, m1_int_time_coefs], axis=1)
    m2_int_time = np.concatenate([times, m2_int_time_coefs], axis=1)
    
    m1_amp_rel = np.concatenate([times, m1_amp_rel_coefs], axis=1)
    m2_amp_rel = np.concatenate([times, m2_amp_rel_coefs], axis=1)
    
    fname_prefix = directory+'mssa_channels_B2/'
    np.savetxt(fname_prefix + 'm0_amp_first_passage_bins_j{}_t16.dat'.format(j_bins), m0_amp)
    np.savetxt(fname_prefix + 'm1_amp_first_passage_bins_j{}_t16.dat'.format(j_bins), m1_amp)
    np.savetxt(fname_prefix + 'm2_amp_first_passage_bins_j{}_t16.dat'.format(j_bins), m2_amp)
    np.savetxt(fname_prefix + 'm1_pitch_first_passage_bins_j{}_t16.dat'.format(j_bins), m1_pitch)
    np.savetxt(fname_prefix + 'm2_pitch_first_passage_bins_j{}_t16.dat'.format(j_bins), m2_pitch)
    np.savetxt(fname_prefix + 'm1_amp_rel_first_passage_bins_j{}_t16.dat'.format(j_bins), m1_amp_rel)
    np.savetxt(fname_prefix + 'm2_amp_rel_first_passage_bins_j{}_t16.dat'.format(j_bins), m2_amp_rel)
    np.savetxt(fname_prefix + 'm1_phase_first_passage_bins_j{}_t16.dat'.format(j_bins), m1_phase)
    np.savetxt(fname_prefix + 'm2_phase_first_passage_bins_j{}_t16.dat'.format(j_bins), m2_phase)
    np.savetxt(fname_prefix + 'm1_int_time_first_passage_bins_j{}_t16.dat'.format(j_bins), m1_int_time)
    np.savetxt(fname_prefix + 'm2_int_time_first_passage_bins_j{}_t16.dat'.format(j_bins), m2_int_time)

def save_coefs_second_passage(directory, channel_file, n_channels, j_bins=30, start_timestep=210, end_timestep=None):
    file = np.loadtxt(directory+channel_file)
    n_times = file.shape[0]
    if end_timestep==None:
        times = np.reshape(file[start_timestep:,0], (n_times-start_timestep,1))
    else:
        times = np.reshape(file[start_timestep:end_timestep,0], (end_timestep-start_timestep,1))
    
    m0_amp_coefs = file[start_timestep:end_timestep,1:n_channels+1]# - 16*5]
    m1_amp_coefs = file[start_timestep:end_timestep,n_channels+1:2*n_channels+1]# - 16*5]
    m2_amp_coefs = file[start_timestep:end_timestep,2*n_channels+1:3*n_channels+1]# - 16*5]
    m1_pitch_coefs = file[start_timestep:end_timestep,3*n_channels+1:4*n_channels+1]# - 16*5]
    m2_pitch_coefs = file[start_timestep:end_timestep,4*n_channels+1:5*n_channels+1]# - 16*5]
    m1_phase_coefs = file[start_timestep:end_timestep,5*n_channels+1:6*n_channels+1]# - 16*5]
    m2_phase_coefs = file[start_timestep:end_timestep,6*n_channels+1:7*n_channels+1]# - 16*5]
    m1_int_time_coefs = file[start_timestep:end_timestep,7*n_channels+1:8*n_channels+1]# - 16*5]
    m2_int_time_coefs = file[start_timestep:end_timestep,8*n_channels+1:9*n_channels+1]# - 16*5]
    
    m1_amp_rel_coefs = m1_amp_coefs/m0_amp_coefs
    m2_amp_rel_coefs = m2_amp_coefs/m0_amp_coefs
    
    m0_amp = np.concatenate([times, m0_amp_coefs], axis=1)
    m1_amp = np.concatenate([times, m1_amp_coefs], axis=1)
    m2_amp = np.concatenate([times, m2_amp_coefs], axis=1)
    m1_pitch = np.concatenate([times, m1_pitch_coefs], axis=1)
    m2_pitch = np.concatenate([times, m2_pitch_coefs], axis=1)
    m1_phase = np.concatenate([times, m1_phase_coefs], axis=1)
    m2_phase = np.concatenate([times, m2_phase_coefs], axis=1)
    m1_int_time = np.concatenate([times, m1_int_time_coefs], axis=1)
    m2_int_time = np.concatenate([times, m2_int_time_coefs], axis=1)
    
    m1_amp_rel = np.concatenate([times, m1_amp_rel_coefs], axis=1)
    m2_amp_rel = np.concatenate([times, m2_amp_rel_coefs], axis=1)
    
    fname_prefix = directory+'mssa_channels_B2/'
    np.savetxt(fname_prefix + 'm0_amp_second_passage_bins_j{}_t16.dat'.format(j_bins), m0_amp)
    np.savetxt(fname_prefix + 'm1_amp_second_passage_bins_j{}_t16.dat'.format(j_bins), m1_amp)
    np.savetxt(fname_prefix + 'm2_amp_second_passage_bins_j{}_t16.dat'.format(j_bins), m2_amp)
    np.savetxt(fname_prefix + 'm1_pitch_second_passage_bins_j{}_t16.dat'.format(j_bins), m1_pitch)
    np.savetxt(fname_prefix + 'm2_pitch_second_passage_bins_j{}_t16.dat'.format(j_bins), m2_pitch)
    np.savetxt(fname_prefix + 'm1_amp_rel_second_passage_bins_j{}_t16.dat'.format(j_bins), m1_amp_rel)
    np.savetxt(fname_prefix + 'm2_amp_rel_second_passage_bins_j{}_t16.dat'.format(j_bins), m2_amp_rel)
    np.savetxt(fname_prefix + 'm1_phase_second_passage_bins_j{}_t16.dat'.format(j_bins), m1_phase)
    np.savetxt(fname_prefix + 'm2_phase_second_passage_bins_j{}_t16.dat'.format(j_bins), m2_phase)
    np.savetxt(fname_prefix + 'm1_int_time_second_passage_bins_j{}_t16.dat'.format(j_bins), m1_int_time)
    np.savetxt(fname_prefix + 'm2_int_time_second_passage_bins_j{}_t16.dat'.format(j_bins), m2_int_time)

In [95]:
save_coefs(directory='../data/', channel_file='B2_for_mssa.dat', 
           n_channels=496, j_bins=30)

In [97]:
save_coefs_first_passage(directory='../data/', channel_file='B2_for_mssa.dat', 
           n_channels=496, j_bins=30)

In [99]:
save_coefs_second_passage(directory='../data/', channel_file='B2_for_mssa.dat', 
           n_channels=496, j_bins=30)